In [ ]:
#| default_exp core

# API

> API for ipykernel-helper

In [ ]:
#| export
from fasthtml.common import *
from fastcore.meta import delegates
from fastcore.utils import patch,dict2obj
from fastcore.docments import sig_source,DocmentText
from fastcore.net import HTTP404NotFoundError
from types import ModuleType, FunctionType, MethodType, BuiltinFunctionType
from inspect import signature, currentframe
from functools import cmp_to_key,partial
from collections.abc import Mapping
from textwrap import dedent
from cloudscraper import create_scraper
from toolslm.funccall import *
from toolslm.xml import *
from ast import literal_eval
from urllib.parse import urlparse, urljoin
from ghapi.all import GhApi

import typing,warnings,re,os,html2text,base64,inspect,traceback

from IPython.core.interactiveshell import InteractiveShell
from IPython.core.completer import ProvisionalCompleterWarning
from jedi import Interpreter, Script as jscript

from IPython.core.display import DisplayObject
from IPython.display import display,Markdown,HTML
from IPython.core.oinspect import Inspector

In [ ]:
from fastcore.test import *
from fastcore.utils import *

In [ ]:
#| export
warnings.filterwarnings('ignore', category=ProvisionalCompleterWarning)

In [ ]:
from pprint import pprint

## InteractiveShell helpers

In [ ]:
#| export
def _safe_repr(obj, max_len=200):
    "Safely get the repr() of an object, truncating if it exceeds max_len."
    try:
        s = str(obj)
        return s[:max_len] + ("…" if len(s)>max_len else "")
    except Exception as e: return f"<repr error: {str(e)}>"

In [ ]:
s = "Some long string that will be truncated"
print(_safe_repr(s, max_len=20))

Some long string tha…


In [ ]:
o = dict(name="Example", data=[1,2,3,4,5] * 5, nested={"a": 1, "b": 2, "c": [3, 4, 5] * 10})
print(_safe_repr(o, max_len=40))

{'name': 'Example', 'data': [1, 2, 3, 4,…


In [ ]:
#| export
@patch
def user_items(self:InteractiveShell, max_len=200, xtra_skip=()):
    "Get user-defined vars & funcs from namespace."
    ns,nsh = self.user_ns,self.user_ns_hidden
    ignore = {'nbmeta', 'receive_nbmeta'}
    ignore.add(xtra_skip)
    rm_types = (
        type, FunctionType, ModuleType, MethodType, BuiltinFunctionType,
        getattr(typing, '_SpecialGenericAlias', ()),
        getattr(typing, '_GenericAlias', ()),
        getattr(typing, '_SpecialForm', ())
    )
    user_items = {k:v for k, v in ns.items()
                  if not k in ignore and k not in nsh}
    user_vars = {k:_safe_repr(v, max_len=max_len)
                 for k, v in user_items.items() if not k.startswith('_') and not isinstance(v, rm_types)}
    user_fns = {k:str(signature(v)) for k, v in user_items.items()
                if isinstance(v, FunctionType) and v.__module__ == '__main__' and not k.startswith('__')}
    return user_vars,user_fns

In [ ]:
# ipy = get_ipython()
# _vs,_fs = ipy.user_items()
# pprint(_vs)
# print('---')
# pprint(_fs)

In [ ]:
#| export
def _rank(c, s):
    "Rank a completion `c` for text `s` with namespace `ns`."
    parts = s.split('.')
    is_public = not c.text.startswith('_')
    if c.type=='param': r=1
    elif c.mod=='__main__': r=2 # local
    elif len(parts)>1 and parts[0]==c.mod: r=3 # module
    elif c.mod=='builtins': r=4
    else: r=5
    return r if is_public else r+0.1

In [ ]:
#| export
@patch
def ranked_complete(self:InteractiveShell, code, line_no=None, col_no=None):
    ns = self.user_ns
    lines = code.splitlines(True)
    if line_no: offset = sum(len(lines[i]) for i in range(line_no-1)) + col_no -1
    else: offset = len(code)
    cs = self.Completer.completions(code, offset)
    def _c(a):
        res = dict2obj({attr: getattr(a, attr) for attr in dir(a) if attr[0]!='_'})
        res['mod']= getattr(ns.get(a.text, None), '__module__', None)
        res['rank'] = _rank(res, s=code)
        return res
    # Remove dunder vars, unless the user seems to be looking for them explicitly
    return [_c(c) for c in cs if not c.text.startswith('__') or '__' in code]

In [ ]:
from random import random

In [ ]:
ipy = get_ipython()

In [ ]:
def range_ex(
    a:str # some param
):
    "some func docstring"
    ...
ipy.ranked_complete('rang')

[{'end': 4,
  'signature': '',
  'start': 0,
  'text': 'range',
  'type': 'class',
  'mod': None,
  'rank': 5},
 {'end': 4,
  'signature': '(a: str)',
  'start': 0,
  'text': 'range_ex',
  'type': 'function',
  'mod': '__main__',
  'rank': 2},
 {'end': 4,
  'signature': '(a, b=None, step=None)',
  'start': 0,
  'text': 'range_of',
  'type': 'function',
  'mod': 'fastcore.basics',
  'rank': 5}]

In [ ]:
res = ipy.ranked_complete('a="foo"\na.', 2, 3)
res[:2]

[{'end': 10,
  'signature': '() -> str',
  'start': 10,
  'text': 'capitalize',
  'type': 'function',
  'mod': None,
  'rank': 5},
 {'end': 10,
  'signature': '() -> str',
  'start': 10,
  'text': 'casefold',
  'type': 'function',
  'mod': None,
  'rank': 5}]

In [ ]:
#| export
def _signatures(ns, s, line, col):
    ctx = Interpreter(s, [ns]).get_signatures(line, col)
    if not ctx: ctx = jscript(s).get_signatures(line, col)
    return ctx

@patch
def sig_help(self:InteractiveShell, code, line_no=None, col_no=None):
    ns = self.user_ns
    ctx = _signatures(ns, code, line=line_no, col=col_no)
    def _s(s): return {'label':s.description,'typ':s.type, 'mod':s.module_name, 'doc':s.docstring(),
                       'idx':s.index, 'params':[{'name':p.name, 'desc':p.description} for p in s.params]}
    return [_s(opt) for opt in ctx]

In [ ]:
s = 'range('
res = ipy.sig_help(s, 1, len(s))
res[0]

{'label': 'class range',
 'typ': 'class',
 'mod': 'builtins',
 'doc': 'range(stop: int)\nrange(start: int, stop: int, step: int=...)\n\nrange(stop) -> range object\nrange(start, stop[, step]) -> range object\n\nReturn an object that produces a sequence of integers from start (inclusive)\nto stop (exclusive) by step.  range(i, j) produces i, i+1, i+2, ..., j-1.\nstart defaults to 0, and stop is omitted!  range(4) produces 0, 1, 2, 3.\nThese are exactly the valid indices for a list of 4 elements.\nWhen step is given, it specifies the increment (or decrement).',
 'idx': 0,
 'params': [{'name': 'stop', 'desc': 'param stop: int'}]}

In [ ]:
#| export
def _maybe_eval(o):
    try: literal_eval(repr(o)); return o
    except: return str(o)

In [ ]:
#| export
@patch
def get_vars(self:InteractiveShell, vs:list, literal=True):
    "Get variables from namespace."
    ns = self.user_ns
    return {v:_maybe_eval(ns[v]) if literal else str(ns[v]) for v in vs if v in ns}

In [ ]:
x, y, fp = 3, 4, open('./00_core.ipynb')
def add(a,b): return a+b

In [ ]:
test_eq(ipy.get_vars(['x', 'y', 'fp']).values(), [x,y,str(fp)])
test_eq(ipy.get_vars(['x', 'y', 'fp'], False).values(), [str(x),str(y),str(fp)])

In [ ]:
#| export
@patch
def eval_exprs(self:InteractiveShell, vs:list, literal=True):
    "Evaluate expressions in namespace."
    ns,res = self.user_ns,{}
    for v in vs:
        try: res[v] = _maybe_eval(eval(v, ns)) if literal else str(eval(v, ns))
        except Exception as e: res[v] = f'<error type="{type(e).__name__}" desc="{e}">\n{traceback.format_exc()}</error>'
    return res

In [ ]:
print(ipy.eval_exprs(['1/0'])['1/0'])

<error type="ZeroDivisionError" desc="division by zero">


Traceback (most recent call last):


  File "/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/ipymini_21011/2788349185.py", line 6, in eval_exprs


    try: res[v] = _maybe_eval(eval(v, ns)) if literal else str(eval(v, ns))


                              ^^^^^^^^^^^


  File "<string>", line 1, in <module>


ZeroDivisionError: division by zero


</error>


In [ ]:
test_eq(ipy.eval_exprs(['x', 'y']), {'x': 3, 'y': 4})
test_eq(ipy.eval_exprs(['add(1,2)']), {'add(1,2)': 3})
test_eq(ipy.eval_exprs(['x+y']), {'x+y': 7})
test_eq(ipy.eval_exprs(['[x, y]']), {'[x, y]': [3, 4]})
test(ipy.eval_exprs(['undefined_var'])['undefined_var'],"NameError: name 'undefined_var' is not defined",operator.contains)

In [ ]:
#| export
def _get_schema(ns: dict, t):
    "Check if tool `t` has errors."
    try: schema = get_schema_nm(t, ns, pname='parameters', evalable=True, skip_hidden=True, dot2dash=True)
    except (KeyError, AttributeError): return f"`{t}` not found. Did you run it?"
    except Exception as e: return f"`{t}`: {e}."
    return {'type':'function', 'function':schema}

@patch
def get_schemas(self:InteractiveShell, fs:list):
    "Get schemas from namespace."
    return {f:_get_schema(self.user_ns, f) for f in fs}

In [ ]:
ipy.get_schemas(['range_ex'])

{'range_ex': {'type': 'function',
  'function': {'name': 'range_ex',
   'description': 'some func docstring',
   'parameters': {'type': 'object',
    'properties': {'a': {'type': 'string', 'description': 'some param'}},
    'required': ['a']}}}}

Errors are passed back as strings:

In [ ]:
def add(a:int,b:int): return a + b
ipy.get_schemas(['add'])

{'add': '`add`: Docstring missing!.'}

In [ ]:
ipy.get_schemas(['div'])

{'div': '`div` not found. Did you run it?'}

Dotted names (like `obj.method`) are supported for getting schemas from object attributes:

In [ ]:
class Calculator:
    def add(self, a:int, b:int) -> int:
        "Add two numbers"
        return a + b

calc = Calculator()
ipy.get_schemas(['calc.add'])

{'calc.add': {'type': 'function',
  'function': {'name': 'calc-add',
   'description': 'Add two numbers\n\nReturns:\n- type: integer',
   'parameters': {'type': 'object',
    'properties': {'a': {'type': 'integer', 'description': ''},
     'b': {'type': 'integer', 'description': ''}},
    'required': ['a', 'b']}}}}

In [ ]:
#| export
@patch
def xpush(self:InteractiveShell, interactive=False, **kw):
    "Like `push`, but with kwargs"
    self.push(kw, interactive=interactive)

In [ ]:
ipy.push(dict(a=2))
a

2

In [ ]:
# ipykernel_helper version uses `**kwargs`
ipy.xpush(a=3)
a

3

The main benefits of using `ipy.push(dict(a=2))` over directly executing code are:

1. **Bulk variable assignment** - You can set multiple variables at once with a single command
2. **Programmatic variable injection** - It provides a way to inject variables into the namespace from another context or function
3. **No execution history** - Variables are added without creating an entry in the execution history
4. **No side effects** - It's a "pure" namespace modification without executing any code that might have side effects

There are several interesting functions in the IPython interpreter object that are useful for notebook development and interactive computing:

1. **`reset`/`reset_selective`** - Clear variables from the namespace (either all or selectively)
2. **`run_cell`/`run_cell_async`** - Execute code in a cell programmatically
3. **`set_next_input`** - Programmatically set the content of the next cell
4. **`system`/`system_raw`/`system_piped`** - Execute shell commands with different output handling
5. **`run_line_magic`/`run_cell_magic`** - Execute IPython magics programmatically
6. **`set_custom_exc`** - Set custom exception handlers

## Displaying MIME data

In [ ]:
cts = '#### A heading\n\nThis is **bold**.'
md_bundle = { 'text/markdown': cts }
ipy.display_pub.publish(data=md_bundle)

In [ ]:
#| export
@patch
def publish(self:InteractiveShell, data='', subtype='plain', mimetype='text', meta=None, update=False, **kw):
    if isinstance(data, DisplayObject): data,_ = self.display_formatter.format(data)
    elif not isinstance(data, Mapping): data = {f'{mimetype}/{subtype}': data}
    self.display_pub.publish(data, metadata=meta, transient=kw, update=update)

In [ ]:
ipy.publish(cts, 'markdown', foo='bar')

In [ ]:
ipy.publish(HTML('<b>hi</b> there'))

In [ ]:
ipy.publish({'text/plain':'hi there'})

In [ ]:
#| export
def transient(data='', subtype='plain', mimetype='text', meta=None, update=False, **kw):
    display({f'{mimetype}/{subtype}': data}, raw=True, metadata=meta, transient=kw, update=update)

In [ ]:
transient('hi there', foo='bar')

hi there

In [ ]:
transient('*hi* **there**', subtype='markdown')

*hi* **there**

In [ ]:
#| export
def run_cmd(cmd, data='', meta=None, update=False, **kw):
    transient(data, meta=meta, update=update, cmd=cmd, **kw)

## read_url et al

In [ ]:
#| export
def _absolutify_imgs(md, base_url):
    def fix(m):
        alt,img_url = m.group(1),m.group(2)
        if not img_url.startswith('http'): img_url = urljoin(base_url, img_url)
        alt = alt.replace('\\','')
        return f'![{alt}]({img_url})'
    return re.sub(r'!\[(.*?)\]\((.*?)\)', fix, md)

In [ ]:
_md = 'An alt text with escape chars\n![\[Uncaptioned image\]](https://www.example.org)'

Running the following will crash solveit:

In [ ]:
# DON'T RUN!
# md_bundle = { 'text/markdown': md}
# ipy.display_pub.publish(data=md_bundle)

This happens because `EscapeSequence` isn't handled correctly inside `FrankenRenderer.render_image` and raises an exception. We fix this by replacing the escape characters in the image alt-text:

In [ ]:
cts = _absolutify_imgs(_md, '')
md_bundle = { 'text/markdown': cts }
ipy.display_pub.publish(data=md_bundle)

In [ ]:
#| export
def get_md(html, url='', mmode=None, ignore_links=False, ignore_images=False, mark_code=True):
    "Convert HTML to markdown with absolute image URLs and optional math mode"
    h = html2text.HTML2Text()
    h.body_width = 0
    h.ignore_links, h.ignore_images, h.mark_code = ignore_links, ignore_images, mark_code
    res = _absolutify_imgs(h.handle(str(html)), url)
    if mmode == 'safe': res = res.replace(r'\\(',r'\(').replace(r'\\)',r'\)')
    return re.sub(r'\[code]\s*\n(.*?)\n\[/code]', lambda m: f'```\n{dedent(m.group(1))}\n```', res, flags=re.DOTALL).strip()

In [ ]:
#| export
def scrape_url(url): 
    o = create_scraper().get(url)
    if not o.encoding or o.encoding == 'ISO-8859-1': o.encoding = 'utf-8'
    return o

In [ ]:
#| eval: false
scrape_url('http://www.example.org').encoding

'utf-8'

In [ ]:
#| export
def _get_math_mode():
    v = os.getenv('USE_KATEX', '')
    if v.lower() in {'0', 'false', 'none', ''}: return None
    return 'dollar' if v.lower().startswith('d') else 'safe'

In [ ]:
#| export
def _aify_imgs(md): return re.sub(r'!\[(.*?)\]\((.*?)\)', r'![\1](\2#ai)', md)

In [ ]:
#| export
def gh_blob_to_raw(url):
    "Convert github.com/user/repo/blob/... URL to raw.githubusercontent.com URL"
    m = re.match(r'https?://(?:www\.)?github\.com/([^/]+)/([^/]+)/blob/([^/]+)/(.+)', url)
    if not m: return url
    owner, repo, ref, path = m.groups()
    return f'https://raw.githubusercontent.com/{owner}/{repo}/{ref}/{path}'

In [ ]:
#| export
def _extract_section(soup, url, selector=None):
    "Extract a specific section from soup, or the whole thing"
    if selector: return '\n\n'.join(str(s) for s in soup.select(selector))
    parsed = urlparse(url)
    if not parsed.fragment: return str(soup)
    section = soup.find(id=parsed.fragment)
    if not section: return ''
    elements = [section]
    current = section.next_sibling
    while current:
        if hasattr(current, 'name') and current.name == section.name: break
        elements.append(current)
        current = current.next_sibling
    return ''.join(str(el) for el in elements)

In [ ]:
#| export
def _convert_math(soup, mode):
    for math in soup.find_all('math'):
        annot = math.find('annotation', {'encoding': 'application/x-tex'})
        if not annot: continue
        tex,display = annot.text.strip(), math.get('display') == 'block'
        if mode == 'dollar': wrap = f'$${tex}$$' if display else f'${tex}$'
        else: wrap = f'$${tex}$$' if display else fr'\({tex}\)'
        math.replace_with(wrap)

In [ ]:
org = 'answerdotai'
reponm = 'dialoghelper'
repopre = f'https://github.com/{org}/{reponm}'

In [ ]:
parse_gh_url(repopre), parse_gh_url(repopre + '/tree/main/dialoghelper')

({'owner': 'answerdotai',
  'repo': 'dialoghelper',
  'typ': None,
  'ref': None,
  'path': None},
 {'owner': 'answerdotai',
  'repo': 'dialoghelper',
  'typ': 'tree',
  'ref': 'main',
  'path': 'dialoghelper'})

In [ ]:
#| export
@llmtool
def read_gh_repo(owner:str, repo:str, ref:str=None, path:str=''):
    "Read GitHub repo info: description, file list, and README"
    api = GhApi()
    info = api.repos.get(owner, repo)
    res = [f"# {info.full_name}", info.description or '']
    ref = ref or info.default_branch
    contents = api.repos.get_content(owner, repo, path or '', ref=ref)
    files = [f"- {'📁 ' if c.type=='dir' else ''}{c.name}" for c in contents]
    res.append(f'\n## /{path or ""} Files\n' + '\n'.join(files))
    if not path:
        try:
            readme = api.repos.get_readme(owner, repo, ref=ref)
            res.append('\n## README\n' + base64.b64decode(readme.content).decode())
        except HTTP404NotFoundError: pass
    return '\n'.join(res)

In [ ]:
#| export
@llmtool
def read_url(
    url:str, # URL to read
    as_md:bool=True, # Convert HTML to markdown
    extract_section:bool=True, # Extract section matching URL fragment or selector
    selector:str=None, # CSS selector to extract specific content
    ai_img:bool=False # Add #ai suffix to image URLs
):
    "Read url from web"
    from bs4 import BeautifulSoup
    gh = parse_gh_url(url)
    if gh:
        if gh['typ']=='blob': url = gh_blob_to_raw(url)
        elif gh['typ'] in (None, 'tree'): return read_gh_repo(gh['owner'], gh['repo'], gh['ref'], gh['path'])
    o = scrape_url(url)
    ctype = (o.headers.get('content-type') or 'text/plain').split(';')[0]
    res = o.text
    if ctype == 'text/html':
        soup = BeautifulSoup(res, 'lxml')
        if ('#' in url and extract_section) or selector: soup = BeautifulSoup(_extract_section(soup, url, selector), 'lxml')
        mmode = _get_math_mode()
        if mmode: _convert_math(soup, mmode)
        base = soup.find('base')
        base_url = urljoin(url, base['href'] if base else '')
        res = get_md(soup, base_url, mmode) if as_md else str(soup)
    if ai_img: res = _aify_imgs(res)
    return res

In [ ]:
print(read_url('https://www.example.org'))

# Example Domain

This domain is for use in documentation examples without needing permission. Avoid use in operations.

[Learn more](https://iana.org/domains/example)


In [ ]:
print(read_url('https://www.example.org', as_md=False, selector='body'))

<html><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>


In [ ]:
print(read_url('https://caddyserver.com/docs/running#unit-files'))

### Unit Files

We provide two different systemd unit files that you can choose between, depending on your use case:

  * [**`caddy.service`**](https://github.com/caddyserver/dist/blob/master/init/caddy.service) if you configure Caddy with a [Caddyfile](/docs/caddyfile). If you prefer to use a different config adapter or a JSON config file, you may override the `ExecStart` and `ExecReload` commands.

  * [**`caddy-api.service`**](https://github.com/caddyserver/dist/blob/master/init/caddy-api.service) if you configure Caddy solely through its [API](/docs/api). This service uses the [`--resume`](/docs/command-line#caddy-run) option which will start Caddy using the `autosave.json` which is [persisted](/docs/json/admin/config/) by default.




They are very similar, but differ in the `ExecStart` and `ExecReload` commands to accommodate the workflows.

If you need to switch between the services, you should disable and stop the previous one before enabling and starting the other. For example

`html2text` removes new lines so we only use `get_md` on html content and static text/markdown content is returned as is with new lines preserved:

In [ ]:
print('\n'.join(read_url('https://fastht.ml/docs/llms.txt').splitlines()[:5]))

# FastHTML

> FastHTML is a python library which brings together Starlette, Uvicorn, HTMX, and fastcore's `FT` "FastTags" into a library for creating server-rendered hypermedia applications. The `FastHTML` class itself inherits from `Starlette`, and adds decorator-based routing with many additions, Beforeware, automatic `FT` to HTML rendering, and much more.

Things to remember when writing FastHTML apps:


Github repos display name, description, top level directory, and readme:

In [ ]:
print(read_url(repopre)[:300])

# AnswerDotAI/dialoghelper
Helper functions for solveit dialogs

## / Files
- 📁 .github
- .gitignore
- CHANGELOG.md
- LICENSE
- MANIFEST.in
- README.md
- 📁 dialoghelper
- 📁 nbs
- pyproject.toml

## README
# dialoghelper

A Python library for programmatic dialog manipulation in [Solveit](https://solv


Github paths show the same as repos, but for that path:

In [ ]:
print(read_url(repopre + '/tree/main/dialoghelper')[:300])

# AnswerDotAI/dialoghelper
Helper functions for solveit dialogs

## /dialoghelper Files
- __init__.py
- _modidx.py
- capture.py
- core.py
- db_dc.py
- screenshot.js
- stdtools.py
- tmux.py
- tracetools.py


Github files are redirected to the raw file:

In [ ]:
print(read_url(repopre + '/blob/main/dialoghelper/core.py')[:300])

# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/00_core.ipynb.

# %% auto #0
__all__ = ['dname_doc', 'md_cls_d', 'dh_settings', 'all_builtins', 'python', 'Placements', 'mermaid_url', 'besure_doc',
           'add_styles', 'find_var', 'set_var', 'find_dname', 'find_msg_id', 'xposta', 'xgeta', 'cal


`read_url` renders latex based on the solveit user's katex setting (`USE_KATEX`)

In [ ]:
os.environ['USE_KATEX']='dollar'
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#S3\.SS2\.SSS1')[:700])

####  3.2.1 Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention" (Figure [2](https://arxiv.org/html/1706.03762v7#S3.F2 "Figure 2 ‣ 3.2.2 Multi-Head Attention ‣ 3.2 Attention ‣ 3 Model Architecture ‣ Attention Is All You Need")). The input consists of queries and keys of dimension $d_{k}$, and values of dimension $d_{v}$. We compute the dot products of the query with all keys, divide each by $\sqrt{d_{k}}$, and apply a softmax function to obtain the weights on the values.

In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix $Q$. The keys and values are also packed together into matrices $K$ a


In [ ]:
os.environ['USE_KATEX']='1'
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#S3\.SS2\.SSS1')[:700])

####  3.2.1 Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention" (Figure [2](https://arxiv.org/html/1706.03762v7#S3.F2 "Figure 2 ‣ 3.2.2 Multi-Head Attention ‣ 3.2 Attention ‣ 3 Model Architecture ‣ Attention Is All You Need")). The input consists of queries and keys of dimension \(d_{k}\), and values of dimension \(d_{v}\). We compute the dot products of the query with all keys, divide each by \(\sqrt{d_{k}}\), and apply a softmax function to obtain the weights on the values.

In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix \(Q\). The keys and values are also packed together into matric


Relative image paths are automatically corrected as well:

In [ ]:
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#Sx1')[:700])

## Attention Visualizations

![Refer to caption](https://arxiv.org/html/x1.png) Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making…more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color.

![Refer to caption](https://arxiv.org/html/x2.png)

![Refer to caption](https://arxiv.org/html/x3.png)

Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution. Top: Full attentions for head 5. Bottom: I


In [ ]:
print(read_url('https://arxiv.org/html/1706.03762v7',selector='#Sx1',ai_img=True)[:700])

## Attention Visualizations

![Refer to caption](https://arxiv.org/html/x1.png#ai) Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making…more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color.

![Refer to caption](https://arxiv.org/html/x2.png#ai)

![Refer to caption](https://arxiv.org/html/x3.png#ai)

Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution. Top: Full attentions for head 5. 


## Other helpers

In [ ]:
#| export
def fix_editable_priority():
    import sys
    from importlib.machinery import PathFinder
    try: sys.meta_path.append(sys.meta_path.pop(sys.meta_path.index(PathFinder)))
    except ValueError: pass

In [ ]:
#| export
@patch
def _repr_markdown_(self:Markdown):
    return f'<div class="prose">\n\n{self.data}\n\n</div>'

## Extension

In [ ]:
#| export
@patch
def _get_info(self:Inspector, obj, oname='', formatter=None, info=None, detail_level=0, omit_sections=()):
    "Custom formatter for ? and ?? output"
    orig = self._orig__get_info(obj, oname=oname, formatter=formatter, info=info,
                                detail_level=detail_level, omit_sections=omit_sections)
    try:
        out = []
        if detail_level==0:
            info_dict = self.info(obj, oname=oname, info=info, detail_level=0)
            out.append(f"```python\n{DocmentText(obj, docstring=False)}\n```")
            if c:=info_dict.get('docstring'): out.append(f'\n\n```\n{c}\n```\n\n')
            if c:=info_dict.get('file'): out.append(f"**File:** `{c}`")
            if c:=info_dict.get('type_name'): out.append(f"**Type:** {c}")
            return {'text/markdown': '\n\n'.join(out), 'text/html': '', 'text/plain': orig['text/plain']}
        info_dict = self.info(obj, oname=oname, info=info, detail_level=2)
        if c:=info_dict.get('source'): out.append(f"\n```python\n{dedent(c)}\n```")
        if c:=info_dict.get('file'): out.append(f"**File:** `{c}`")
        return {'text/markdown': '\n\n'.join(out), 'text/html': '', 'text/plain': orig['text/plain']}
    except Exception: return orig

In [ ]:
flexiclass?

```python
def flexiclass(
    cls, # The class to convert
)->dataclass:

```



```
Convert `cls` into a `dataclass` like `make_nullable`. Converts in place and also returns the result.
```



**File:** `~/aai-ws/fastcore/fastcore/xtras.py`

**Type:** function

In [ ]:
flexiclass??


```python
def flexiclass(
        cls # The class to convert
    ) -> dataclass:
    "Convert `cls` into a `dataclass` like `make_nullable`. Converts in place and also returns the result."
    if is_dataclass(cls): return make_nullable(cls)
    for k,v in get_annotations_ex(cls)[0].items():
        if not hasattr(cls,k) or getattr(cls,k) is MISSING:
            setattr(cls, k, field(default=UNSET))
    return dataclass(cls, init=True, repr=True, eq=True, order=False, unsafe_hash=False, frozen=False)
```

**File:** `~/aai-ws/fastcore/fastcore/xtras.py`

In [ ]:
def f(a:int=0 # aa
): pass

@delegates(f)
def g(
    b:int, # bb
    **kwargs
)->int: # Returns the meaning of life
    "The g function"
    # nothing to see here
    pass

In [ ]:
g?

```python
def g(
    b:int, # bb
    a:int=0, # aa
)->int: # Returns the meaning of life

```



```
The g function
```



**File:** `/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/ipykernel_60639/3905198878.py`

**Type:** function

In [ ]:
g??


```python
@delegates(f)
def g(
    b:int, # bb
    **kwargs
)->int: # Returns the meaning of life
    "The g function"
    # nothing to see here
    pass
```

**File:** `/var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/ipykernel_60639/3905198878.py`

In [ ]:
#| export
@patch
async def run_cell_magic(self:InteractiveShell, magic_name, line, cell):
    result = self._orig_run_cell_magic(magic_name, line, cell)
    if inspect.iscoroutine(result): result = await result
    if isinstance(result, FT): result = HTML(to_xml(result))
    return result

def _await_cell_magic(lines):
    if lines and 'get_ipython().run_cell_magic(' in lines[0]: lines = ['await ' + lines[0]] + lines[1:]
    return lines

def load_ipython_extension(ip):
    from ipykernel_helper import transient,run_cmd

    ns = ip.user_ns
    ns['read_gh_repo'], ns['read_url'],ns['transient'],ns['run_cmd'] = read_gh_repo,read_url,transient,run_cmd
    ip.input_transformer_manager.line_transforms.append(_await_cell_magic)

## export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()